# Lab B — The VRAM Wall

**Week 2 · GPU Infrastructure for LLMs** · run on the **2× RTX 5090** server (WSL2)

In the lecture, VRAM was the first wall: a model only runs if **weights + KV cache + activations + overhead** fit in the GPU's memory. This lab makes that real. You will:

1. Predict a model's memory from the formula, then **measure** it.
2. Watch the **KV cache grow** as context grows.
3. **Hit an out-of-memory error on purpose** — feel the hard 32 GB ceiling.
4. **Quantize** the model (FP16 → 4-bit) and watch it fit.
5. Build a **"will it fit?"** budget calculator.

> This runs on the local server, **not Colab** — it needs the real GPUs.

## 0. Requirements

You need PyTorch built for **Blackwell (sm_120)** — that means **CUDA 12.8+ / PyTorch ≥ 2.7**. If `torch.cuda.is_available()` is False or you get a `sm_120 not supported` error, reinstall torch with the cu128 wheels.

```
pip install --upgrade "torch>=2.7" --index-url https://download.pytorch.org/whl/cu128
pip install --upgrade transformers accelerate bitsandbytes
```

In [ ]:
import torch, time, gc, subprocess
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPUs:", torch.cuda.device_count())
assert torch.cuda.is_available(), "No CUDA GPU visible — check your driver / torch build"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB  compute sm_{p.major}{p.minor}")

In [ ]:
# --- helpers we reuse all lab ---
def gpu_report(tag=""):
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        used = (total - free) / 1e9
        print(f"  [{tag}] GPU{i}: {used:5.1f} GB used / {total/1e9:.1f} GB total   ({free/1e9:5.1f} GB free)")

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

gpu_report("idle")   # should be ~0 GB used if nothing else is running

## 1. The memory formula

$$\text{VRAM} \approx \underbrace{P \times b}_{\text{weights}} + \underbrace{\text{KV cache}}_{\text{grows with context}} + \text{activations} + \text{overhead}$$

- **P** = number of parameters, **b** = bytes per parameter.

| Precision | bytes / param | 7B weights | 70B weights |
|---|---|---|---|
| FP16 / BF16 | 2 | 14 GB | 140 GB |
| FP8 | 1 | 7 GB | 70 GB |
| INT4 / NF4 | 0.5 | 3.5 GB | 35 GB |

**One 5090 = 32 GB.** Already you can see a 70B model in FP16 (140 GB) has no chance on one card — that's the wall we're about to hit.

## 2. Measure a real model's weight footprint

Set `MODEL_ID` to a model that is **already downloaded on the box** (or that you're happy to download). A 7–8B model is the sweet spot for a 32 GB card in FP16.

In [ ]:
# ===== CONFIG — set this once =====
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # <-- replace with a model available on your server
# ==================================

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

free_gpu()
before = torch.cuda.mem_get_info(0)[0]           # free bytes before load
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map={"": 0})
after  = torch.cuda.mem_get_info(0)[0]           # free bytes after load
measured_gb = (before - after) / 1e9

n_params = sum(p.numel() for p in model.parameters())
predicted_gb = n_params * 2 / 1e9                # FP16 = 2 bytes/param

print(f"Parameters : {n_params/1e9:.2f} B")
print(f"Predicted  : {predicted_gb:.1f} GB  (params x 2 bytes)")
print(f"Measured   : {measured_gb:.1f} GB")
print(f"Overhead   : {measured_gb - predicted_gb:.1f} GB  (CUDA context, buffers)")
gpu_report("model loaded")

## 3. Watch the KV cache grow

The KV cache stores one key + value vector **per token, per layer** — so it grows linearly with context length and batch size. Let's feed longer and longer prompts and watch VRAM climb *above* the fixed weight cost.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
base_used = (torch.cuda.mem_get_info(0)[1] - torch.cuda.mem_get_info(0)[0]) / 1e9

print(f"{'context tokens':>15} | {'extra VRAM (KV+act)':>20}")
for ctx in [256, 1024, 4096, 8192]:
    free_gpu()
    ids = torch.randint(0, tok.vocab_size, (1, ctx), device="cuda:0")
    with torch.no_grad():
        out = model(ids, use_cache=True)      # builds the KV cache for `ctx` tokens
    used = (torch.cuda.mem_get_info(0)[1] - torch.cuda.mem_get_info(0)[0]) / 1e9
    print(f"{ctx:>15} | {used - base_used:>18.2f} GB")
    del out
free_gpu()

**Takeaway:** weights are a fixed cost; the **KV cache is the variable cost** that eats your remaining headroom as prompts get longer or batches get bigger. This is why long-context serving runs out of memory even when the model itself fits.

## 4. Hit the wall — OOM on purpose

Let's allocate memory in 2 GB chunks until the GPU refuses. This is the **hard 32 GB ceiling** — there is no "swap to disk"; it just fails.

In [ ]:
free_gpu()
blocks = []
try:
    while True:
        # 1024^3 float16 = ~2.1 GB per block
        blocks.append(torch.empty(1024, 1024, 1024, dtype=torch.float16, device="cuda:0"))
        used = (torch.cuda.mem_get_info(0)[1] - torch.cuda.mem_get_info(0)[0]) / 1e9
        print(f"  allocated block {len(blocks):2d}  ->  {used:5.1f} GB used")
except torch.cuda.OutOfMemoryError as e:
    print("\n===> HIT THE VRAM WALL")
    print("   ", str(e).splitlines()[0][:160])
finally:
    del blocks
    free_gpu()
    gpu_report("after OOM cleanup")

## 5. Quantize to fit

Same model, fewer bytes per weight. **4-bit (NF4)** drops the weights to ~0.5 bytes/param — about **4× smaller** than FP16 — which is the most common trick to squeeze a model onto a smaller card.

In [ ]:
# free the FP16 model first so we can compare cleanly
try:
    del model
except NameError:
    pass
free_gpu()
gpu_report("before 4-bit load")

from transformers import BitsAndBytesConfig
try:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16)
    before = torch.cuda.mem_get_info(0)[0]
    model4 = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0})
    after  = torch.cuda.mem_get_info(0)[0]
    print(f"\n4-bit weights: {(before-after)/1e9:.1f} GB   (vs ~{measured_gb:.1f} GB in FP16)")
    gpu_report("4-bit loaded")

    # prove it still generates
    ids = tok("In one sentence, what is a GPU?", return_tensors="pt").to("cuda:0")
    out = model4.generate(**ids, max_new_tokens=40, do_sample=False)
    print("\nSample output:", tok.decode(out[0], skip_special_tokens=True))
except Exception as e:
    print("bitsandbytes 4-bit failed on this setup:", repr(e)[:200])
    print("If bnb doesn't support sm_120 yet, try load_in_8bit=True, or torchao / native FP8.")

## 6. "Will it fit?" — the budget calculator

Tie it all together: given a model, precision, and workload, does it fit on **one 32 GB 5090**?

In [ ]:
def vram_estimate_gb(params_b, bytes_per_param, n_layers, hidden, n_heads, n_kv_heads,
                     seq, batch, kv_bytes=2):
    head_dim = hidden // n_heads
    weights  = params_b * 1e9 * bytes_per_param
    kv_cache = 2 * n_layers * n_kv_heads * head_dim * seq * batch * kv_bytes
    overhead = 0.10 * weights + 2e9          # CUDA context + activations, rough
    total    = weights + kv_cache + overhead
    return total / 1e9

def fits(name, **kw):
    g = vram_estimate_gb(**kw)
    print(f"  {name:28s} -> {g:6.1f} GB   {'✅ fits on 32 GB' if g <= 32 else '❌ over the wall'}")

# Example: a 7B-class model (Llama/Qwen-ish geometry), 4k context, batch 1
GEO = dict(n_layers=32, hidden=4096, n_heads=32, n_kv_heads=8, seq=4096, batch=1)
print("7B model, 4k context, batch 1:")
fits("  FP16 (2 B/param)",  params_b=7,  bytes_per_param=2.0, **GEO)
fits("  FP8  (1 B/param)",  params_b=7,  bytes_per_param=1.0, **GEO)
fits("  INT4 (0.5 B/param)",params_b=7,  bytes_per_param=0.5, **GEO)
print("\n70B model, 4k context, batch 1:")
fits("  FP16", params_b=70, bytes_per_param=2.0, **dict(GEO, n_layers=80, hidden=8192, n_heads=64, n_kv_heads=8))
fits("  INT4", params_b=70, bytes_per_param=0.5, **dict(GEO, n_layers=80, hidden=8192, n_heads=64, n_kv_heads=8))

## Reflection (write your answers)

1. How close was the **measured** weight VRAM to the **predicted** value? What is the extra overhead?
2. At what context length did the KV cache start to dominate your free memory?
3. Exactly how many GB did 4-bit save vs FP16, and did the sample output still look reasonable?
4. Using the calculator: what is the **largest model** (and precision) you can serve at 8k context, batch 1, on **one** 5090? On **both** (64 GB, if you could split it)?

### Cleanup

In [ ]:
for name in ["model", "model4", "out", "ids"]:
    if name in dir():
        try: exec(f"del {name}")
        except Exception: pass
free_gpu()
gpu_report("clean")